In [1]:
# Sesión 4 del módulo 8, Clase 28-08
# Ejemplo 1: Regularización con Dropout
# Dropout apaga aleatoriamente neuronas durante el entrenamiento, para evitar que la red memorice los datos y aprenda patrones más generales.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


ModuleNotFoundError: No module named 'tensorflow'

In [5]:
import tensorflow as tf
import time

a = tf.random.normal([3000, 3000])
start = time.time()
for _ in range(10):
    _ = tf.matmul(a, a)
print("Tiempo:", time.time()-start, "segundos")

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:

# Dataset Pima Indians Diabetes (ejemplo clásico)
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
cols = ["Pregnancies","Glucose","BloodPressure","SkinThickness","Insulin",
        "BMI","DiabetesPedigree","Age","Outcome"]
df = pd.read_csv(url, names=cols)
print(df)
# Se carga el dataset de diabetes desde un repositorio público.
# Tiene 768 registros de pacientes con variables médicas:
# Ejemplo: Glucose (nivel de glucosa), Age (edad), BMI (índice de masa corporal).
# Outcome es la variable objetivo (0 = no tiene diabetes, 1 = tiene diabetes).

X = df.drop("Outcome", axis=1).values
y = df["Outcome"].values
# X: características de entrada (todas menos la columna Outcome).
# y: la columna de salida (Outcome), es decir la clase binaria a predecir.

# Escalar datos
sc = StandardScaler()
X = sc.fit_transform(X)
# # Problema: las variables tienen escalas distintas (ej. glucosa ≈ 100, edad ≈ 30, insulina puede ser 0–800).
# Solución: StandardScaler transforma los datos a media = 0 y desviación estándar = 1.
# Ejemplo: si glucosa promedio es 100 con desviación 15 → un valor 130 se convierte en (130-100)/15 = 2.
# Esto hace que la red neuronal aprenda más rápido y de forma estable.

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
# Separamos 80% entrenamiento y 20% prueba.
# random_state=42 → semilla para reproducibilidad.
# stratify=y → asegura que el balance de clases (0 y 1) sea parecido en train y test.

In [ ]:
# Modelo con Dropout
model = keras.Sequential([
    layers.Dense(32, activation="relu", input_shape=(X.shape[1],)),
    layers.Dropout(0.5),   # apaga el 50% de neuronas en cada paso
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])
# Capa 1 – Dense(32, ReLU):
# 32 neuronas totalmente conectadas.
# activation="relu" → función de activación Rectified Linear Unit:
# f(x)=max(0,x)
# Permite capturar relaciones no lineales.
# input_shape=(X.shape[1],) → el número de características de entrada (en el dataset de diabetes son 8).

# Capa 2 – Dropout(0.5):
# Durante el entrenamiento, “apaga” aleatoriamente el 50% de las neuronas de la capa anterior en cada iteración.
# Esto evita que el modelo se memorice los datos y ayuda a generalizar mejor.
# Ejemplo:
# Es como si en un examen le taparas a un estudiante la mitad de sus apuntes, para obligarlo a razonar y no recitar de memoria.

# Capa 3 – Dense(16, ReLU):
# Otra capa oculta, más pequeña (16 neuronas).
# Refina la representación de los datos.

# Capa 4 – Dense(1, Sigmoid):
# Capa de salida con una sola neurona.
# activation="sigmoid" transforma la salida en un valor entre 0 y 1:
# σ(x)=1/(1+e−x​)
# Interpretable como probabilidad de tener diabetes (si ≥0.5 → clase 1, si <0.5 → clase 0).

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
# optimizer="adam" → algoritmo adaptativo que ajusta los pesos automáticamente (una mejora del SGD).
# loss="binary_crossentropy" → función de pérdida para clasificación binaria:
# L=(−1/n)​∑[ylog(p^​)+(1−y)log(1−p^​)]
# Castiga más cuando la red está muy segura y se equivoca.
# metrics=["accuracy"] → se medirá el % de aciertos (0 o 1 correctos).
history = model.fit(X_train, y_train, epochs=50, batch_size=32,
                    validation_split=0.2, verbose=0)
# X_train, y_train → datos de entrenamiento.
# epochs=50 → la red verá todo el dataset 50 veces completas.
# batch_size=32 → entrena en mini-lotes de 32 ejemplos cada vez.
# validation_split=0.2 → el 20% de los datos de entrenamiento se aparta automáticamente para validación (para monitorear overfitting).
# verbose=0 → no imprime la barra de progreso, pero history guarda toda la información de las curvas de pérdida y accuracy.

In [ ]:
# Graficar curvas
plt.plot(history.history["accuracy"], label="train_acc")
plt.plot(history.history["val_accuracy"], label="val_acc")
plt.xlabel("Épocas"); plt.ylabel("Accuracy")
plt.title("Regularización con Dropout")
plt.legend(); plt.show()
# history es un objeto que guarda cómo evolucionaron las métricas en cada época.
# history.history es un diccionario con listas:
# "accuracy" → accuracy en datos de entrenamiento por época.
# "val_accuracy" → accuracy en datos de validación por época.
# También podría incluir "loss" y "val_loss" (la función de pérdida).

# Dibuja dos curvas:
# Una azul (train_acc) → desempeño en entrenamiento.
# Otra naranja (val_acc) → desempeño en validación.

# xlabel("Épocas") → eje X son las épocas (1, 2, 3 …).
# ylabel("Accuracy") → eje Y muestra la precisión (0.0 a 1.0).

# title("Regularización con Dropout") → título de la gráfica.
# legend() → muestra la leyenda (train_acc vs val_acc).
# show() → despliega el gráfico.

Lo que se ve en el gráfico

Eje X (horizontal): número de épocas (de 0 a 50).

Eje Y (vertical): accuracy (entre 0.55 y 0.80 aprox).

Curva azul (train_acc): precisión en el conjunto de entrenamiento.

Curva naranja (val_acc): precisión en el conjunto de validación.

Interpretación

Inicio del entrenamiento (épocas 0–10):

El modelo empieza con baja precisión (0.55).

Rápidamente sube hasta alrededor de 0.70–0.75 en pocas épocas.

Esto indica que la red aprende rápido al comienzo.

Entre épocas 10 y 30:

Tanto entrenamiento como validación mejoran y se mantienen muy cercanas.

Significa que el modelo generaliza bien: no memoriza solo el train, sino que también funciona en datos nuevos.

Entre épocas 30 y 50:

El accuracy de validación llega hasta ~0.79–0.80.

El de entrenamiento se mantiene parecido (~0.77–0.78).

La diferencia entre train y val es pequeña → Dropout funcionó bien evitando sobreajuste.

Conclusión
El modelo con Dropout alcanzó una precisión cercana al 80% en validación.
No hay sobreajuste fuerte, porque las curvas de entrenamiento y validación son muy similares.

El Dropout ayudó a mantener el modelo más robusto: sin él, lo normal sería que train_acc >> val_acc.

El azul es lo que el modelo aprendió en los datos que ya conocía. El naranja es cómo se comporta en datos nuevos. Como ambas curvas están muy cerca, significa que nuestro modelo está aprendiendo de forma general y no solo memorizando.”

In [ ]:
# Ejemplo 2: EarlyStopping
# Detiene el entrenamiento cuando la red deja de mejorar en validación, evitando sobreajuste (antes de que la red empiece a memorizar).
early = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)
# monitor="val_loss" → observa la pérdida (loss) en el conjunto de validación.
# patience=5 → si pasan 5 épocas seguidas sin mejora en val_loss, detiene el entrenamiento.
# restore_best_weights=True → al terminar, restaura los pesos de la mejor época (la que tuvo menor val_loss).
# En pocas palabras: si el modelo empieza a memorizar (sobreajustar) y la validación deja de mejorar, EarlyStopping corta el entrenamiento
# antes de gastar tiempo innecesario.

model_es = keras.Sequential([
    layers.Dense(32, activation="relu", input_shape=(X.shape[1],)),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])
# Capa 1: 32 neuronas ReLU, con forma de entrada (X.shape[1],) (en este dataset son 8 características).
# Capa 2: 16 neuronas ReLU.
# Capa 3: 1 neurona con Sigmoid → salida binaria (probabilidad de tener diabetes).
# Este modelo es similar al anterior, pero sin Dropout.

model_es.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
history_es = model_es.fit(X_train, y_train, epochs=100, batch_size=32,
                          validation_split=0.2, callbacks=[early], verbose=0)
# optimizer="adam" → optimizador adaptativo eficiente.
# loss="binary_crossentropy" → pérdida apropiada para clasificación binaria.
# metrics=["accuracy"] → mide el % de aciertos.

# Se pide entrenar hasta 100 épocas, pero…
# Gracias a EarlyStopping, el entrenamiento se detendrá antes si val_loss no mejora después de 5 épocas consecutivas.
# validation_split=0.2 → el 20% de los datos de entrenamiento se usa para validación.

print("Épocas usadas:", len(history_es.history["loss"]))
# Esto imprime cuántas épocas realmente se usaron.
# Aunque pedimos epochs=100, el entrenamiento se detiene en la época 33.

Si hubieras dejado entrenar 100 épocas, probablemente el modelo habría empezado a memorizar los datos de entrenamiento (sobreajuste).

Con EarlyStopping, el modelo se “auto-corrige”:

Aprende lo suficiente.

Se detiene antes de sobreajustar.

Conserva los mejores pesos.

Es como decirle a un estudiante:
“Deja de repasar cuando ya alcanzaste tu punto máximo, porque seguir memorizando solo te hace confundir”.

In [ ]:
# Evaluación con métricas
# No basta con accuracy: también usamos precisión, recall y F1-score.
# Evaluación en test
y_prob = model_es.predict(X_test).ravel()
# model_es.predict(X_test) → devuelve la probabilidad de que cada muestra sea clase 1 (en este caso, tener diabetes).
# Ejemplo de salida: [0.23, 0.89, 0.65, ...].
# .ravel() → aplana el array a 1D para que quede más fácil de manejar.
y_pred = (y_prob >= 0.5).astype(int)
# Se aplica un umbral de 0.5:
# Si la probabilidad ≥ 0.5 → clase 1.
# Si la probabilidad < 0.5 → clase 0.
# astype(int) → convierte los booleanos (True/False) en enteros (1/0).

# Ejemplo:
# Probabilidades: [0.23, 0.89, 0.65]
# Predicciones: [0, 1, 1]

print("Matriz de confusión:")
print(confusion_matrix(y_test, y_pred))

print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred, digits=3))

[[83 17]
 [23 31]]
Fila 1 (clase 0: no diabético):
83 → verdaderos negativos (TN): el modelo dijo 0 y era 0.
17 → falsos positivos (FP): el modelo dijo 1 pero era 0.

Fila 2 (clase 1: diabético):
23 → falsos negativos (FN): el modelo dijo 0 pero era 1.
31 → verdaderos positivos (TP): el modelo dijo 1 y era 1.

En resumen:
Detectó bien 83 casos sanos y 31 casos de diabetes.

Falló en 17 casos que marcó diabéticos sin serlo y en 23 casos de diabéticos que no detectó.

2. Reporte de clasificación
Clase 0 (no diabético)
Precision = 0.783 → De todos los que predijo como no diabéticos, acertó en el 78.30%.

Recall = 0.830 → De todos los no diabéticos reales, encontró el 83%.
F1-score = 0.806 → Promedio balanceado entre precisión y recall.
Support = 100 → Había 100 ejemplos de clase 0 en el test.

Clase 1 (diabético)
Precision = 0.646 → De todos los que predijo como diabéticos, acertó en el 64.6%.
Recall = 0.573 → De todos los diabéticos reales, solo detectó el 57.3%.
F1-score = 0.608 → Un poco más bajo que la clase 0, porque es más difícil de detectar.
Support = 54 → Había 54 ejemplos de clase 1 en el test.

3. Promedios
Accuracy = 0.740 (74%)
Globalmente, acertó aproximadamente en 3 de cada 4 casos.

Macro avg: promedio simple entre las clases.
Muestra que en promedio el modelo anda entre 70–72% en precisión y recall.
Weighted avg: promedio ponderado por la cantidad de ejemplos de cada clase.
Como hay más clase 0 que clase 1, este promedio se parece más a los números de la clase 0.

Interpretación final
El modelo predice bastante bien la clase 0 (no diabético), pero le cuesta un poco más con la clase 1 (diabético).

Esto es común en datasets desbalanceados: hay menos ejemplos de la clase minoritaria, y el modelo tiende a equivocarse más allí.

Accuracy global de 74% → buen inicio, pero podría mejorarse con técnicas como balanceo de clases, ajuste de umbral o redes más profundas.

In [ ]:
# Más ejemplos
# ==============================================================================
# SESIÓN 4: ENTRENAMIENTO Y OPTIMIZACIÓN DE REDES NEURONALES
# Este código demuestra cómo entrenar un modelo, usar un optimizador y aplicar
# técnicas de regularización como Dropout.
# ==============================================================================

# 1. Importar las librerías necesarias
# ------------------------------------
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
import matplotlib.pyplot as plt

print("Librerías importadas correctamente.")

In [ ]:
# 2. Cargar y preparar el conjunto de datos Fashion MNIST
# --------------------------------------------------------
# El conjunto de datos Fashion MNIST consiste en 70,000 imágenes a color gris y en baja resolución, de 28x28 píxeles. Estas imágenes están divididas en:
# 60,000 imágenes de entrenamiento.
# 10,000 imágenes de prueba.
# Cada imagen pertenece a una de 10 categorías de ropa.
# Variables del conjunto de datos
# El conjunto de datos tiene dos variables principales:
# Imágenes (train_images, test_images): Estas son las variables de entrada de la red neuronal.
# Tipo de variable: Son variables numéricas que representan la intensidad de cada píxel de la imagen. Los valores van de 0 a 255.
# Forma de los datos: Cada imagen es un arreglo bidimensional de 28x28 píxeles. Cuando se carga en un modelo de Deep Learning, se le añade una dimensión extra para el canal de color (en este caso, 1 para blanco y negro),
# por lo que la forma final es (28, 28, 1).
# Etiquetas (train_labels, test_labels): Estas son las variables de salida que el modelo debe aprender a predecir.
# Tipo de variable: Son variables categóricas, representadas como números enteros del 0 al 9. Cada número corresponde a una de las 10 categorías de ropa.
# Las 10 categorías (etiquetas) son:
# 0: Camiseta/top
# 1: Pantalón
# 2: Sudadera
# 3: Vestido
# 4: Abrigo
# 5: Sandalia
# 6: Camisa
# 7: Zapatilla deportiva
# 8: Bolso
# 9: Bota al tobillo
# .
(train_images, train_labels), (test_images, test_labels) = keras.datasets.fashion_mnist.load_data()
# Esta línea descarga y carga el conjunto de datos de Fashion MNIST directamente desde la biblioteca Keras. Divide los datos en dos grupos:

# train_images y train_labels: Son 60,000 imágenes y sus etiquetas correspondientes que se usan para el entrenamiento del modelo. El modelo
# aprenderá de estos datos.

# test_images y test_labels: Son 10,000 imágenes y sus etiquetas que se usan para la evaluación del modelo después del entrenamiento. Sirven para
# probar qué tan bien generaliza el modelo a datos que nunca ha visto.

# Normalizar los valores de los píxeles (de 0-255 a 0-1) para facilitar el entrenamiento.
train_images, test_images = train_images / 255.0, test_images / 255.0
# Las imágenes se cargan con valores de píxeles que van de 0 a 255, donde 0 es negro y 255 es blanco. Las redes neuronales funcionan mejor cuando
# los valores de entrada son pequeños, generalmente entre 0 y 1.
# ¿Por qué se normalizan? Al dividir cada valor de píxel entre 255 (el valor máximo), se transforma el rango de 0-255 a un rango de 0.0-1.0. Esto
# ayuda a que el proceso de entrenamiento sea más estable y rápido.

# Redimensionar las imágenes para que se ajusten a la entrada de la red convolucional.
# La forma (shape) es (número de imágenes, alto, ancho, canales de color).
# Aquí, 1 canal es para imágenes en blanco y negro.
train_images = train_images.reshape((60000, 28, 28, 1))
test_images = test_images.reshape((10000, 28, 28, 1))
# Cuando los datos se cargan inicialmente, las imágenes tienen la forma de (60000, 28, 28). La red neuronal convolucional (CNN) necesita una
# dimensión adicional que represente el canal de color de la imagen.

# El reshape lo que hace es cambiar la forma del arreglo de datos para que sea (número_de_imágenes, alto, ancho, canales_de_color).
# Para Fashion MNIST, que son imágenes en blanco y negro, solo hay un canal de color, por lo que el valor es 1.
# Esto transforma la forma de los datos de entrenamiento a (60000, 28, 28, 1) y los datos de prueba a (10000, 28, 28, 1). Con esta nueva forma,
# los datos están listos para ser procesados por las capas convolucionales del modelo.

print("Datos cargados y normalizados.")

In [ ]:
# 3. Construir el modelo de red neuronal (incluyendo una capa de regularización)
# ----------------------------------------------------------------------------
# Utilizaremos una red neuronal convolucional (CNN) con una capa de Dropout.
# Dropout es una técnica para reducir el sobreajuste (overfitting).
# Durante el entrenamiento, 'apaga' aleatoriamente algunas neuronas de una capa,
# lo que obliga a la red a no depender de una sola neurona para aprender.
model = keras.Sequential([
    # Capa Convolucional 1: Extrae características de las imágenes.
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),

    # Capa de Max Pooling 1: Reduce la dimensionalidad de los datos.
    layers.MaxPooling2D((2, 2)),

    # Capa de Regularización (Dropout)
    # Dropout "apaga" el 25% de las neuronas de la capa anterior de forma aleatoria en cada paso.
    layers.Dropout(0.25),

    # Capa Convolucional 2: Extrae más características.
    layers.Conv2D(64, (3, 3), activation='relu'),

    # Capa de Max Pooling 2: Sigue reduciendo la dimensionalidad.
    layers.MaxPooling2D((2, 2)),

    # Aplanar los datos para pasarlos a las capas densas.
    layers.Flatten(),

    # Capa Densa: 64 neuronas para procesar las características.
    layers.Dense(64, activation='relu'),

    # Capa de Salida: 10 neuronas (una para cada clase de ropa).
    # La función Softmax convierte los resultados en probabilidades.
    layers.Dense(10, activation='softmax')
])

# 1. keras.Sequential
# Esto define que el modelo será una secuencia de capas. Es la forma más simple y común de construir una red neuronal, donde las capas se apilan una tras otra.

# 2. layers.Conv2D (Capa Convolucional)
# ¿Qué hace? Esta es la capa fundamental de una Red Neuronal Convolucional (CNN). Su trabajo es extraer características de las imágenes, como bordes,
# texturas o formas.
# 32: Es el número de "filtros" o "kernels" que se usan. Cada filtro aprende a detectar una característica diferente en la imagen. En este caso, la capa
# extrae 32
# características distintas.
# (3, 3): Es el tamaño del filtro. Un filtro de 3x3 es una pequeña ventana de 3 por 3 píxeles que "barre" toda la imagen para detectar patrones.
# activation='relu': La función de activación ReLU introduce no linealidad y ayuda al modelo a aprender patrones más complejos.
# input_shape=(28, 28, 1): Define la forma de la entrada para la primera capa. Le dice al modelo que espere imágenes de 28 píxeles de alto, 28 de ancho y
# con 1 canal de color (para blanco y negro).

# 3. layers.MaxPooling2D (Capa de Pooling)
# ¿Qué hace? Reduce la dimensionalidad de los datos. Toma la salida de la capa convolucional y "comprime" la información. Por ejemplo, en una ventana de 2x2,
# solo mantiene el valor más alto, descartando los demás.

# ¿Por qué es útil? Esto ayuda a reducir la cantidad de parámetros y, por lo tanto, el tiempo de cálculo. Además, hace que el modelo sea más robusto a
# pequeñas variaciones en la posición de las características en la imagen.

# 4. layers.Dropout (Capa de Regularización)
# ¿Qué hace? Es una técnica de regularización que previene el sobreajuste (overfitting). El sobreajuste ocurre cuando el modelo memoriza los datos de
# entrenamiento en lugar de aprender los patrones generales.
# ¿Cómo funciona? Durante el entrenamiento, Dropout "apaga" aleatoriamente una fracción de las neuronas de la capa anterior.

# 0.25: Indica que el 25% de las neuronas se "apagarán" de manera aleatoria en cada paso de entrenamiento. Esto obliga a la red a no depender de ninguna
# neurona en particular, lo que la hace más robusta.

# 5. layers.Flatten (Capa de Aplanamiento)
# ¿Qué hace? Transforma la salida de las capas convolucionales (que es un tensor 2D) en un arreglo unidimensional (un vector). Este paso es necesario
# para conectar las capas convolucionales con las capas densas tradicionales.

# 6. layers.Dense (Capa Densa)
# ¿Qué hace? Esta es una capa completamente conectada. Cada neurona de esta capa recibe una entrada de todas las neuronas de la capa anterior.

# 64: En este caso, la capa tiene 64 neuronas.

# activation='relu': Se usa la misma función de activación.

# 7. layers.Dense (Capa de Salida)
# ¿Qué hace? Esta es la capa final que produce el resultado de la clasificación.

# 10: El número de neuronas es 10 porque hay 10 clases de ropa que el modelo debe clasificar.

# activation='softmax': Esta función de activación convierte los valores de salida de las 10 neuronas en un conjunto de probabilidades que suman 1.
# La clase con la probabilidad más alta es la predicción del modelo.

print("\nArquitectura del modelo construida.")
model.summary()

La tabla proporciona una vista detallada de cada capa de tu red, desde la entrada hasta la salida.

Layer (type): Muestra el nombre y el tipo de cada capa. Puedes ver que tienes capas Conv2D (Convolucionales), MaxPooling2D (Max Pooling), Dropout y Dense (densas). Cada capa realiza una operación específica en los datos de la imagen. Por ejemplo, Conv2D extrae características y MaxPooling2D las reduce.

Output Shape: Indica la forma de los datos a medida que pasan por cada capa.

conv2d: La forma de salida es (None, 26, 26, 32). El None es el tamaño del lote de datos, que es flexible. Los datos de 28x28 píxeles se reducen a 26x26 después de pasar por el filtro de 3x3 de la capa convolucional. Los 32 son el número de filtros o características que la capa ha extraído.

max_pooling2d: La forma se reduce a (None, 13, 13, 32). La capa de pooling, con un tamaño de (2, 2), reduce la altura y el ancho de la imagen a la mitad.

flatten: Transforma los datos de la última capa de pooling, que tenían una forma de (5, 5, 64), en un arreglo unidimensional de 1600 valores (5 x 5 x 64 = 1600).

Param # (Número de Parámetros): Es el número de parámetros entrenables (pesos y sesgos) que tiene cada capa.

conv2d: Los 320 parámetros provienen de los 32 filtros de 3x3 píxeles, más un sesgo para cada filtro. La fórmula es (tamaño_filtro_ancho * tamaño_filtro_alto * canales_entrada + 1) * número_filtros, lo que da (3 * 3 * 1 + 1) * 32 = 320.

max_pooling2d y dropout: Estas capas no tienen parámetros entrenables (Param # = 0), ya que su función es simplemente reducir o "apagar" neuronas sin aprender.

flatten: Tampoco tiene parámetros, ya que su única función es cambiar la forma de los datos.

dense: Los 102,464 parámetros provienen de la conexión de las 1600 entradas de la capa Flatten a las 64 neuronas de esta capa densa, más los 64 sesgos. El cálculo es (1600 * 64) + 64 = 102,464.

dense_1: Es la capa de salida. Conecta las 64 neuronas de la capa anterior a las 10 neuronas de salida. El cálculo es (64 * 10) + 10 = 650.

Total params: La suma total de los parámetros de todas las capas, que es 121,930. Esto te da una idea del tamaño del modelo.

Trainable params: Indica cuántos de esos parámetros se ajustarán durante el entrenamiento. En este caso, todos lo son.

Este resumen te permite ver de manera clara cómo la red neuronal procesa y transforma los datos a través de sus capas para finalmente realizar una clasificación.

In [ ]:
# 4. Compilar el modelo
# ---------------------
# Aquí definimos cómo se entrenará el modelo.
# - 'optimizer': El algoritmo que ajustará los pesos (ej. 'adam', un optimizador muy eficiente).
# - 'loss': La función de pérdida que mide el error (ej. 'sparse_categorical_crossentropy' para este problema).
# - 'metrics': Las métricas que se mostrarán durante el entrenamiento (ej. 'accuracy' para la precisión).
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
# El Optimizador (optimizer='adam')
# ¿Qué es? El optimizador es el algoritmo que ajustará los pesos de tu red neuronal durante el entrenamiento para que la red pueda aprender de los datos.

# 'adam': Es un optimizador muy popular y eficiente. Se basa en el Descenso de Gradiente (un concepto que ya se ha visto), pero de una manera más inteligente.
# En lugar de usar la misma tasa de ajuste para todos los pesos, Adam adapta la tasa de aprendizaje para cada parámetro individualmente, lo que acelera el
# entrenamiento y mejora el rendimiento.

# . La Función de Pérdida (loss='sparse_categorical_crossentropy')
# ¿Qué es? La función de pérdida (o loss function) es una medida de qué tan bien se está desempeñando tu modelo. Es una forma de calcular el error entre
# las predicciones del modelo y las etiquetas reales.

# 'sparse_categorical_crossentropy': Este tipo de función de pérdida es el más adecuado para los problemas de clasificación multicategoría, como el que se
# tiene con Fashion MNIST, donde las etiquetas son números enteros (0, 1, 2, ..., 9). Esta función mide la diferencia entre las probabilidades predichas por
# el modelo para cada clase y la clase real.

# 3. Las Métricas (metrics=['accuracy'])
# ¿Qué es? Las métricas son las herramientas que se usan para monitorear el proceso de entrenamiento y la evaluación del modelo. A diferencia de la función
# de pérdida, que el modelo busca minimizar internamente, las métricas son solo para que tú las observes y entiendas el rendimiento.

# 'accuracy': Esta métrica mide la precisión del modelo, es decir, el porcentaje de imágenes que el modelo clasificó correctamente. Durante el entrenamiento,
# Keras imprimirá el valor de la precisión en cada época, lo que te permite ver cómo mejora el modelo con el tiempo.

In [ ]:
# 5. Entrenar el modelo
# ---------------------
# El modelo aprende a partir de los datos de entrenamiento.
# - 'epochs': El número de veces que el modelo recorrerá todo el conjunto de datos.
# - 'validation_data': El conjunto de datos de prueba se usa para validar el modelo después
#   de cada época y detectar el sobreajuste.
print("\n--- Iniciando el entrenamiento ---")
history = model.fit(train_images,
                    train_labels,
                    epochs=10,
                    validation_data=(test_images, test_labels))

print("\nEntrenamiento finalizado.")

# La función model.fit() es el corazón del entrenamiento en Keras. Le dices al modelo qué datos usar para aprender y cómo debe ser el proceso.

# history = model.fit(...): El método fit devuelve un objeto history que contiene información valiosa sobre el entrenamiento, como la pérdida y
# la precisión en cada época.
# Esta información se usa para crear los gráficos que te mostré anteriormente.

# train_images y train_labels: Son el conjunto de datos de entrenamiento. La red neuronal utiliza estas 60,000 imágenes para ajustar sus pesos y
# sesgos. Este es el input (las imágenes) y el target (las etiquetas correctas) que la red debe aprender a relacionar.

# epochs=10: Una época es una iteración completa sobre todo el conjunto de datos de entrenamiento. Al poner epochs=10, le estás diciendo al modelo
# que recorra todas las 60,000 imágenes 10 veces. Aumentar el número de épocas generalmente mejora el rendimiento del modelo, pero también aumenta
# el riesgo de sobreajuste (overfitting).

# validation_data=(test_images, test_labels): Esto es muy importante. Keras utiliza este conjunto de datos para validar el modelo al final de cada
# época. La validación te permite monitorear el desempeño de la red en datos que nunca ha visto antes. Es la clave para detectar el sobreajuste.

# ¿Cómo funciona el proceso?
# Iteración 1 (Época 1): El modelo toma los datos de train_images y train_labels, realiza el forward pass (la propagación hacia adelante), calcula
# el loss (el error) y luego ajusta los pesos con el backpropagation (la retropropagación) y el optimizador.

# Al finalizar la primera época, Keras evalúa el modelo con el conjunto de validation_data (los datos de prueba) para darte una idea de cómo se
# está comportando en datos nuevos.

# Iteraciones 2 a 10: Este proceso se repite para cada una de las 10 épocas. En cada iteración, el modelo se vuelve un poco más preciso y el
# error disminuye.

# El objetivo es que tanto la pérdida de entrenamiento como la pérdida de validación disminuyan, lo que indica que el modelo está aprendiendo
# de forma efectiva y que puede generalizar a datos nuevos.

Epoch 1/10: Indica que el modelo está en la Época 1 de un total de 10 épocas.

1875/1875: Esto significa que el modelo ha procesado los 1,875 lotes (batches) de datos de entrenamiento en esta época.

accuracy: 0.7417 - loss: 0.7053: Estos valores corresponden al desempeño del modelo en los datos de entrenamiento.

accuracy: La precisión en el conjunto de entrenamiento. En la primera época, el modelo clasificó correctamente el 74.17% de las imágenes.

loss: La pérdida en el conjunto de entrenamiento. La pérdida mide el error del modelo. Un valor de 0.7053 en la primera época es un punto de partida.

val_accuracy: 0.8491 - val_loss: 0.4158: Estos valores corresponden al desempeño del modelo en los datos de validación (el conjunto de prueba).

val_accuracy: La precisión en el conjunto de validación. Es una métrica crucial para saber qué tan bien el modelo generaliza a datos que no ha visto. En la primera época, la precisión es del 84.91%.

val_loss: La pérdida en el conjunto de validación.

¿Qué nos dicen estos resultados?
La clave es observar cómo cambian estos valores a lo largo de las épocas:

Pérdida (loss y val_loss): Ambos valores, tanto de entrenamiento como de validación, disminuyen con cada época. Esto es una excelente señal, ya que significa que el modelo está aprendiendo de forma efectiva y que el optimizador está ajustando los pesos correctamente para minimizar el error.

Precisión (accuracy y val_accuracy): Ambos valores aumentan a medida que avanza el entrenamiento.

Fíjense que la precisión de validación es consistentemente alta, lo que indica que el modelo no solo está aprendiendo de los datos de entrenamiento, sino que también está generalizando muy bien a los datos nuevos.

Si la pérdida de validación comenzara a subir en algún momento, mientras que la pérdida de entrenamiento sigue bajando, sería una señal de sobreajuste (overfitting). En este caso, no se observa ese problema, lo que demuestra la efectividad de la arquitectura y el dropout.

In [ ]:
# 6. Visualización de resultados del entrenamiento
# ------------------------------------------------
# Graficar la pérdida y la precisión para ver cómo aprendió el modelo.
# Esto ayuda a detectar si hay sobreajuste.
# Si la 'val_loss' (pérdida de validación) empieza a subir mientras la 'loss' (pérdida de entrenamiento)
# sigue bajando, es una señal de sobreajuste.

plt.figure(figsize=(10, 5))

# Gráfica de la pérdida
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Pérdida en entrenamiento')
plt.plot(history.history['val_loss'], label='Pérdida en validación')
plt.title('Pérdida (Loss) del Modelo')
plt.ylabel('Pérdida')
plt.xlabel('Época')
plt.legend()

# Gráfica de la precisión
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Precisión en entrenamiento')
plt.plot(history.history['val_accuracy'], label='Precisión en validación')
plt.title('Precisión (Accuracy) del Modelo')
plt.ylabel('Precisión')
plt.xlabel('Época')
plt.legend()

plt.show()

Gráfico 1: Pérdida (Loss) del Modelo
Este gráfico muestra la evolución de la pérdida (o error) del modelo a lo largo del entrenamiento.

Eje horizontal (Época): Representa las épocas de entrenamiento, que son las veces que el modelo ha procesado todo el conjunto de datos. En tu caso, el entrenamiento duró 10 épocas.

Eje vertical (Pérdida): Muestra el valor de la función de pérdida. Cuanto más bajo sea este valor, mejor.

Línea azul (Pérdida en entrenamiento): Esta curva muestra la pérdida del modelo en los datos que está usando para aprender. Como puedes ver, esta línea baja consistentemente, lo que significa que el modelo está aprendiendo de manera efectiva.

Línea naranja (Pérdida en validación): Esta curva muestra la pérdida del modelo en los datos de prueba (test_images), que el modelo no ha visto durante el entrenamiento. Es una métrica crucial para saber si el modelo es capaz de generalizar a datos nuevos.

En este gráfico, ambas curvas disminuyen a un ritmo similar, lo que es una excelente señal. Demuestra que el modelo no solo está aprendiendo de los datos de entrenamiento, sino que también está generalizando bien a datos nuevos, y no hay un problema de sobreajuste.

Gráfico 2: Precisión (Accuracy) del Modelo
Este gráfico muestra la evolución de la precisión del modelo a lo largo del entrenamiento.

Eje horizontal (Época): Al igual que en el gráfico anterior, representa las épocas de entrenamiento.

Eje vertical (Precisión): Muestra el porcentaje de predicciones correctas del modelo. Cuanto más alto sea el valor (más cerca de 1.0), mejor.

Línea azul (Precisión en entrenamiento): Esta curva muestra la precisión del modelo en los datos de entrenamiento. Como es de esperar, la precisión aumenta constantemente.

Línea naranja (Precisión en validación): Esta curva muestra la precisión en los datos de validación.

En tu gráfico, ambas curvas aumentan, lo que significa que la capacidad del modelo para clasificar las imágenes correctamente está mejorando tanto en los datos de entrenamiento como en los datos que no ha visto. Los valores finales (alrededor de 0.93 para entrenamiento y 0.91 para validación) son muy buenos.

En resumen, estos dos gráficos demuestran que el modelo fue entrenado con éxito, que aprendió de manera efectiva y que es capaz de generalizar y hacer buenas predicciones en datos nuevos.

In [ ]:
# 7. Evaluación final del modelo
# ------------------------------
# Evaluar el rendimiento del modelo en los datos de prueba.
print("\n--- Evaluación en los datos de prueba ---")
test_loss, test_acc = model.evaluate(test_images,  test_labels, verbose=2)
print(f"Precisión final en los datos de prueba: {test_acc:.4f}")

In [ ]:
# 8. Realizando una predicción
# ----------------------------
# Usar el modelo entrenado para predecir la clase de la primera imagen de prueba.
# Primero, obtenemos la imagen y la añadimos a un nuevo batch de 1.
imagen_para_predecir = test_images[0:1]
predicciones = model.predict(imagen_para_predecir)
# Primero, el código toma la primera imagen del conjunto de prueba (test_images[0:1]). Es importante que el modelo no haya visto esta imagen durante
# el entrenamiento. Luego, esta imagen se pasa a través del modelo ya entrenado con model.predict(), que produce un array de 10 números, uno para cada
# clase de ropa.

# La predicción es un array de 10 probabilidades. La clase con la probabilidad más alta
# es la predicción del modelo.
predicted_label = tf.argmax(predicciones[0]).numpy()
true_label = test_labels[0]
# La salida del modelo son probabilidades para cada clase. Por ejemplo, podría decir que la probabilidad de que sea un Trouser es 0.95, y el resto
# de las probabilidades son muy bajas. La función tf.argmax() encuentra el índice de la probabilidad más alta. Este índice es la predicción del modelo.
# Luego, el código simplemente toma la etiqueta real de la imagen (test_labels[0]) para compararla con la predicción.

class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

print(f"\nPredicción del modelo: {class_names[predicted_label]}")
print(f"Etiqueta real de la imagen: {class_names[true_label]}")
# Finalmente, usa un array con los nombres de las clases (class_names) para mostrar un resultado legible en lugar de solo un número. Por ejemplo, en
# lugar de 7, imprime Sneaker. Esto es muy útil para validar a simple vista si el modelo acertó en su predicción.

# Mostrar la imagen que se predijo
plt.figure()
plt.imshow(imagen_para_predecir.reshape(28, 28), cmap=plt.cm.binary)
plt.title(f"Imagen: {class_names[true_label]}")
plt.show()

Predicción del modelo: Ankle boot: Esta línea muestra el resultado de la predicción que el modelo hizo para la imagen. Después de procesar la imagen de prueba, la red neuronal calculó las probabilidades para cada una de las 10 categorías y eligió la categoría con la probabilidad más alta, que en este caso fue "Ankle boot" (bota al tobillo).

Etiqueta real de la imagen: Ankle boot: Esta línea te dice cuál es la categoría verdadera de la imagen, según el conjunto de datos.

El hecho de que la predicción del modelo y la etiqueta real de la imagen sean idénticas significa que el modelo clasificó la imagen correctamente. Este es el objetivo final del entrenamiento: que el modelo pueda hacer una predicción precisa en datos que nunca ha visto antes.

Otro ejemplo
El Dataset Pima Indians Diabetes es un conjunto de datos muy famoso en machine learning biomédico. Fue recopilado por el National Institute of Diabetes and Digestive and Kidney Diseases (NIDDK) y su objetivo es predecir si una mujer de origen Pima (población indígena en EE. UU.) tiene diabetes o no, usando variables clínicas.

Variables del dataset

Tiene 768 registros y 9 columnas:

Pregnancies → Número de embarazos de la paciente.

Tipo: numérico (entero).

Ejemplo: 6.

Glucose → Nivel de glucosa en plasma a las 2 horas de una prueba oral de tolerancia a la glucosa.

Tipo: numérico (mg/dL).

Ejemplo: 148.

BloodPressure → Presión arterial diastólica (mm Hg).

Tipo: numérico.

Ejemplo: 72.

SkinThickness → Espesor del pliegue cutáneo del tríceps (mm).

Tipo: numérico.

Ejemplo: 35.

Insulin → Nivel de insulina sérica a las 2 horas (mu U/ml).

Tipo: numérico.

Ejemplo: 0 o 150.

BMI (Body Mass Index) → Índice de masa corporal:

BMI=peso(kg)/altura(m)^2

Tipo: numérico.

Ejemplo: 33.6.

DiabetesPedigreeFunction → Un puntaje que mide la predisposición genética a la diabetes.

Tipo: numérico (función calculada).

Ejemplo: 0.627.

Age → Edad de la paciente en años.

Tipo: numérico (entero).

Ejemplo: 50.

Outcome → Variable objetivo.

Tipo: binaria.

0 = no tiene diabetes.

1 = tiene diabetes.

Resumen para estudiantes

Entrada (X): 8 variables médicas y personales (embarazos, glucosa, presión, BMI, etc.).

Salida (y): un valor binario → ¿tiene diabetes (1) o no (0)?

In [ ]:
# Ejemplo A: Overfitting vs Underfitting
# Mostrar qué pasa cuando una red es muy simple (subajuste) o muy compleja (sobreajuste).
# Ejemplo A: Underfitting vs Overfitting
from tensorflow.keras import layers, Sequential
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Dataset de diabetes
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
cols = ["Pregnancies","Glucose","BloodPressure","SkinThickness","Insulin",
        "BMI","DiabetesPedigree","Age","Outcome"]
df = pd.read_csv(url, names=cols)

# Variables X (entradas) y y (salida)
X = df.drop("Outcome", axis=1).values
y = df["Outcome"].values

# Normalizar
sc = StandardScaler()
X = sc.fit_transform(X)

# División train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Modelo muy simple (underfitting)
model_simple = Sequential([
    layers.Dense(2, activation="relu", input_shape=(X.shape[1],)),
    layers.Dense(1, activation="sigmoid")
])

model_simple.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
hist_simple = model_simple.fit(X_train, y_train, epochs=50, batch_size=32,
                               validation_split=0.2, verbose=0)

# Modelo muy complejo (posible overfitting)
model_complex = Sequential([
    layers.Dense(128, activation="relu", input_shape=(X.shape[1],)),
    layers.Dense(64, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

model_complex.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
hist_complex = model_complex.fit(X_train, y_train, epochs=50, batch_size=32,
                                 validation_split=0.2, verbose=0)

In [ ]:
# Graficar
import matplotlib.pyplot as plt
plt.plot(hist_simple.history["val_accuracy"], label="Simple (val)")
plt.plot(hist_complex.history["val_accuracy"], label="Complejo (val)")
plt.xlabel("Épocas"); plt.ylabel("Accuracy validación")
plt.title("Underfitting vs Overfitting")
plt.legend(); plt.show()

Lo que muestra el gráfico

Eje X (horizontal): épocas de entrenamiento (de 1 a 50).

Eje Y (vertical): accuracy en validación (qué tan bien predice en datos no vistos).

Curva azul (Simple val): desempeño de un modelo muy simple (pocas neuronas).

Curva naranja (Complejo val): desempeño de un modelo más grande (más neuronas y capas).

Modelo Simple (azul)

Parte con un accuracy de ~0.50 (lo mismo que adivinar al azar en un problema binario).

Sube lentamente hasta ~0.65.

No logra capturar la complejidad del problema.

Esto se llama Underfitting (subajuste):

El modelo es demasiado limitado → no aprende lo suficiente.

Modelo Complejo (naranja)

Parte con mejor rendimiento (~0.75) y rápidamente sube a ~0.83–0.84.

Después se mantiene oscilando en ese rango.

Capta patrones más complejos y generaliza mejor.

Sin embargo, si siguieras entrenando muchas más épocas, podría empezar a memorizar el set de entrenamiento y sobreajustar.

Enseñanza:

Un modelo muy simple = underfitting → aprende poco, baja accuracy.

Un modelo más complejo = mejor capacidad → logra buen accuracy en validación.

Balance ideal: un modelo lo suficientemente complejo, pero con regularización (Dropout, EarlyStopping, etc.) para evitar el sobreajuste.

Es como:

Estudiante modelo simple: estudia muy por encima, no entiende bien → reprueba.

Estudiante modelo complejo: estudia bien, pero si memoriza sin entender → se confunde en el examen (overfitting).

In [ ]:
# Ejemplo B: Regularización L2
# Mostrar cómo usar penalización L2 para que los pesos no crezcan demasiado.
# Se usa el dataset Pima Indians Diabetes (8 variables médicas → salida: 0 = no diabetes, 1 = diabetes).

# El modelo tiene 3 capas densas (32 → 16 → 1).
# En cada capa densa aplicamos Regularización L2 para controlar el tamaño de los pesos.
# Esto ayuda a reducir overfitting, porque el modelo no se “confía” en unos pocos pesos grandes, sino que distribuye mejor la importancia entre las neuronas.
# Es como decirle al estudiante:
# “No dependas de memorizar una sola fórmula gigante, reparte tu estudio en varios temas pequeños”.
from tensorflow.keras import regularizers

model_l2 = Sequential([
    layers.Dense(32, activation="relu", input_shape=(X.shape[1],),
                 kernel_regularizer=regularizers.l2(0.01)),  # penalización
    layers.Dense(16, activation="relu", kernel_regularizer=regularizers.l2(0.01)),
    layers.Dense(1, activation="sigmoid")
])

model_l2.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
hist_l2 = model_l2.fit(X_train, y_train, epochs=50, batch_size=32,
                       validation_split=0.2, verbose=0)
# Explicación:
# La regularización L2 añade un “castigo” a los pesos grandes → el modelo prefiere soluciones más simples y generalizables.
# Es como ponerle un freno al modelo para que no dependa demasiado de ciertos parámetros.
# Este bloque implementa Regularización L2 en una red neuronal con Keras.

# Regularización L2 (Ridge): agrega una penalización al costo/pérdida para que los pesos del modelo no crezcan demasiado.
# La idea: si los pesos son muy grandes → la red puede volverse inestable o memorizar (overfitting).
# Con L2, el modelo “prefiere” pesos pequeños y distribuidos → lo hace más robusto y generalizable.

In [ ]:
# Ejemplo C: Learning Rate y su efecto
# Ver qué pasa cuando usamos distintos valores de learning rate.
# Learning rate alto
# Se trabaja con el dataset de diabetes (Pima Indians), el mismo que cargaste antes (X_train, y_train).
# Se construyen dos redes idénticas, pero la única diferencia es el valor de la tasa de aprendizaje (learning_rate).
model_lr_high = Sequential([
    layers.Dense(32, activation="relu", input_shape=(X.shape[1],)),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])
model_lr_high.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.1),
                      loss="binary_crossentropy", metrics=["accuracy"])
hist_lr_high = model_lr_high.fit(X_train, y_train, epochs=30, batch_size=32,
                                 validation_split=0.2, verbose=0)
# Red neuronal con 2 capas ocultas.
# Usa Adam como optimizador.
# Aquí el learning_rate=0.1 es muy alto.
# Significado: en cada actualización, los pasos para ajustar los pesos son muy grandes.
# Ventaja: puede aprender rápido.
# Riesgo: inestabilidad → la pérdida puede oscilar y no converger bien.

# Learning rate bajo
model_lr_low = Sequential([
    layers.Dense(32, activation="relu", input_shape=(X.shape[1],)),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])
# Misma arquitectura.
# Ahora learning_rate=0.0001, muy pequeño.
# Significado: los pasos son muy pequeños.
# Ventaja: aprendizaje más estable y preciso.
# Desventaja: es mucho más lento → necesita más épocas para converger.
model_lr_low.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
                     loss="binary_crossentropy", metrics=["accuracy"])
hist_lr_low = model_lr_low.fit(X_train, y_train, epochs=30, batch_size=32,
                               validation_split=0.2, verbose=0)
# Ambos modelos entrenan 30 épocas.
# Se guarda el historial (hist_lr_high y hist_lr_low) para comparar luego loss y accuracy.

# Learning rate alto (0.1):
# Sube rápido la accuracy al inicio.
# Puede quedarse estancado o incluso degradar la validación → riesgo de no converger.
# Learning rate bajo (0.0001):
# Avanza lento, casi plano en pocas épocas.
# Pero si entrenamos muchas épocas, puede llegar a buen resultado con más estabilidad.

# La enseñanza clave es:
# El learning rate es como la longitud del paso al subir una montaña:
# Si el paso es muy grande → te puedes pasar de la cima.
# Si el paso es muy pequeño → llegas, pero tardas muchísimo.
# Lo ideal es un valor intermedio (p. ej. 0.001 para Adam).

In [ ]:
# Graficar
plt.plot(hist_lr_high.history["val_loss"], label="LR alto")
plt.plot(hist_lr_low.history["val_loss"], label="LR bajo")
plt.xlabel("Épocas"); plt.ylabel("Val Loss")
plt.title("Efecto del Learning Rate")
plt.legend(); plt.show()

Interpretación del gráfico
Curva azul (LR alto = 0.1)
Baja muy rápido al inicio, llegando enseguida a valores bajos de pérdida.

Sin embargo, la curva es muy inestable: sube y baja constantemente.
Esto refleja que con un learning rate alto la red da pasos grandes → a veces se acerca a la solución, pero luego se “pasa de largo” y la pérdida oscila.
Ventaja: aprendizaje rápido.
Riesgo: inestabilidad, puede no converger bien.

Curva naranja (LR bajo = 0.0001)

Disminuye muy lentamente y de forma suave.

No presenta oscilaciones bruscas → el entrenamiento es más estable.

Pero incluso después de 30 épocas, la pérdida sigue bastante alta comparada con LR alto.
Ventaja: estabilidad.
Desventaja: aprendizaje demasiado lento, necesitaríamos muchísimas más épocas para llegar a buen rendimiento.

Enseñanza para tus estudiantes

LR alto = pasos grandes → rápido pero arriesgado.

LR bajo = pasos pequeños → estable pero lento.

Lo ideal es un punto intermedio (p. ej., 0.001 en Adam), que combine velocidad y estabilidad.

Una analogía:
Imagina subir una montaña:

Si das zancadas enormes (LR alto), avanzas rápido, pero te puedes pasar de la cima.

Si das pasos diminutos (LR bajo), llegas con precisión, pero tardas mucho.

Lo mejor es un ritmo intermedio.